# 사내 RAGaaS — 수강생 실습본

원본: `src/raw/Chapter01/03-rag/05_ragaas.py`  
대응 슬라이드: `docs/raw/Chapter01/day01/1-4-rag-theory-and-core-modules.pdf`

**학습 목표:** PDF 텍스트 → 업로드 구성 → RRF 검색 → 답변의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** 로컬 PDF를 읽고 RAG 검색과 LLM 호출을 한다. RAG_PROVIDER=samsung이면 사내 API, mock이면 모의 응답을 쓴다. 실습본의 업로드·삭제 요청은 주석 처리되어 있다.

**특별 주의:** RAG_PROVIDER=samsung이면 사내망·rag_credential_key·rag_api_key와 두 인증 헤더가 필요하다. mock이면 RAG HTTP 요청은 보내지 않지만 LLM 호출은 계속된다. 업로드·검색·삭제 단계의 실행 여부를 확인한다.



## 흐름을 먼저 읽기

PDF 텍스트 → 업로드 구성 → RRF 검색 → 답변

핵심은 문서 준비·검색·답변 단계의 경계이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: PDF 텍스트 → 업로드 구성 → RRF 검색 → 답변. 인덱스·두 인증 헤더·검색 응답은 어디서 확인하나? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 노트북 경로 설정

원본은 `__file__`로 PDF 위치를 찾는다. 노트북에서는 정의되지 않을 수 있으므로 원본 코드셀보다 먼저 이 설정 셀을 수동 실행한다. 프로젝트 루트를 찾지 못하면 명시적으로 오류가 난다. 원본 코드는 수정하지 않는다.


In [ ]:
from pathlib import Path
_cwd = Path.cwd().resolve()
_roots = [p for p in (_cwd, *_cwd.parents) if (p / 'pyproject.toml').is_file()]
if not _roots:
    raise FileNotFoundError('pyproject.toml이 있는 프로젝트 폴더에서 노트북을 열어 주세요.')
_project_root = _roots[0]
__file__ = str(_project_root / 'src/raw/Chapter01/03-rag/05_ragaas.py')
if not Path(__file__).is_file():
    raise FileNotFoundError(f'원본 Python 파일을 찾을 수 없습니다: {__file__}')


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** 로컬 PDF를 읽고 RAG 검색과 LLM 호출을 한다. RAG_PROVIDER=samsung이면 사내 API, mock이면 모의 응답을 쓴다. 실습본의 업로드·삭제 요청은 주석 처리되어 있다.

**특별 주의:** RAG_PROVIDER=samsung이면 사내망·rag_credential_key·rag_api_key와 두 인증 헤더가 필요하다. mock이면 RAG HTTP 요청은 보내지 않지만 LLM 호출은 계속된다. 업로드·검색·삭제 단계의 실행 여부를 확인한다.



In [ ]:
"""
RAG as a Service (RAGaaS): 삼성 사내 RAG API 활용

기존 방식 vs RAGaaS 비교:
  기존: 청킹 → 임베딩 클래스 구현 → 벡터스토어 생성 → 검색
  RAGaaS: 문서 업로드(API 1회) → 검색(API 1회)
  청킹·임베딩·벡터스토어 관리를 API가 대신 처리한다.

사외에서는 RAG_PROVIDER=mock으로 요청·응답 형식을 연습한다.
실제 문서 내용이 필요한 실습은 사내망에서 RAG_PROVIDER=samsung으로 실행한다.

실행:
  python 03-rag/05_ragaas.py
"""
import os
from pathlib import Path

from dotenv import load_dotenv

from config.llm import get_llm
from config.ragaas import post_ragaas

load_dotenv()

RAG_PROVIDER = os.getenv('RAG_PROVIDER', 'samsung').strip().lower()
RAG_API_KEY = os.getenv('rag_api_key')
INDEX_NAME = os.getenv('index_name') or ('mock-pet-insurance' if RAG_PROVIDER == 'mock' else '')
if not INDEX_NAME:
    raise RuntimeError('사내 RAGaaS에는 index_name 환경변수가 필요합니다.')
# 실제 사내 요청의 api-key 헤더에는 RAG_API_KEY가 들어간다.
print(f'RAGaaS 모드: {RAG_PROVIDER}')


# ── 1부: 문서 업로드 ──────────────────────────────────────────────────────────
# insert-doc 엔드포인트 하나로 청킹·임베딩·벡터스토어 저장을 모두 처리한다.
# chunk_factor: 청킹 방식과 크기를 서버에 위임 (직접 구현 불필요)

# 업로드할 텍스트 준비 (예: 파일을 읽어서 문자열로 전달)
PDF_PATH = Path(__file__).parent / 'data' / 'meritz_pet_insurance.pdf'

# pdfminer 로 텍스트 추출 (RAGaaS는 raw text를 받음)
from langchain_community.document_loaders import PDFMinerLoader
docs = PDFMinerLoader(str(PDF_PATH.resolve())).load()
raw_text = '\n'.join(doc.page_content for doc in docs)

payload = {
    'index_name': INDEX_NAME,
    'data': {
        'doc_id': 'meritz_pet_insurance',
        'title': '메리츠 펫보험 약관',
        'content': raw_text,
    },
    'chunk_factor': {
        'logic': 'fixed_size',
        'chunk_size': 500,
        'chunk_overlap': 75,
        'separator': ' ',
    },
}

print('=== 문서 업로드 ===')
# TODO: insert-doc 엔드포인트로 문서를 업로드해 주세요.
# response = post_ragaas('insert-doc', payload, api_key=RAG_API_KEY)
# print(f'상태: {response.status_code}')
# print(response.text[:200])
# print()


# ── 2부: 문서 검색 ────────────────────────────────────────────────────────────
# retrieve-* 엔드포인트: bm25(키워드), knn(의미), rrf(하이브리드) 중 선택

question = '펫보험 보장 내용이 뭐야?'

search_payload = {
    # INDEX를 기반으로 문서를 검색한다.
        # 따라서, 문서를 등록할 때, 하나의 문서 말고도
        # 다양한 그러나 하나의 범주에 속할 수 있는 문서들을
        # 잘 전처리 했다면 (RAG화 하기 적절하게)
        # 동일한 INDEX_NAME으로 여러 문서들을 등록해서 사용할 수도 있다.
    'index_name': INDEX_NAME,
    'permission_groups': ['rag-public'],
    'query_text': question,
    'num_result_doc': 3,
    # 'fields_exclude': ['v_merge_title_content'],
}

print('=== 문서 검색 (RRF: 하이브리드) ===')

# TODO: retrieve-rrf 엔드포인트로 관련 문서를 검색해 주세요.

response = post_ragaas('retrieve-rrf', search_payload, api_key=RAG_API_KEY)
result = response.json()
# print(result)
hits = result['hits']['hits']
context_list = [hit['_source']['merge_title_content'] for hit in hits]
context = '\n\n'.join(context_list)
print(f'검색된 문서 수: {len(hits)}')
# print(context_list)


# ── 3부: RAG 체인 ─────────────────────────────────────────────────────────────
# 검색 결과를 context로 주입해 LLM이 문서 기반으로 답변하도록 한다.
# get_llm(): 로컬은 OpenAI, 사내망은 삼성 LLM으로 자동 전환


from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = get_llm()

# 모델에게 입력할 지시사항과, 사용자의 질문을 간단하게 정의
prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.'),
    ('human', 'context:\n{context}\n\nquestion:\n{question}'),
    # ai -> model의 답변을 관리하는 방식
])

chain = prompt | llm | StrOutputParser()

print(f'=== 질문: {question} ===')
for chunk in chain.stream({'context': context, 'question': question}):
    print(chunk, end='', flush=True)
print('\n')


# ── 4부: 문서 삭제 (선택) ─────────────────────────────────────────────────────
# 실습 종료 후 업로드한 인덱스를 정리한다.

delete_payload = {
    'index_name': INDEX_NAME,
    'permission_groups': ['rag-public'],
    'doc_id': 'meritz_pet_insurance',
}

# TODO: delete-doc 엔드포인트로 업로드한 문서를 삭제해 주세요.
# response = post_ragaas('delete-doc', delete_payload, api_key=RAG_API_KEY)
# print(f'=== 문서 삭제: {response.status_code} ===')


## 결과를 검토하기

- 예상 동작: 검색된 문서 수와 문서 기반 답변을 출력한다. 실습본에서는 업로드·삭제 요청이 주석 처리되어 있다.
- 확인 질문: 인덱스·두 인증 헤더·검색 응답은 어디서 확인하나?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
